# E791 $D^+\to\pi^-\pi^+\pi^+$ — Fit 2 fit example

Same Fit 2 model as notebook 1. The $\rho(770)$ coefficient is fixed to $1+0i$; all other `RealImag` coefficients float. Only $m_{\rho(1450)}$ and $\Gamma_{\rho(1450)}$ float among dynamical parameters. All free fit parameters are randomized inside their bounds before the minimization.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, weighted_resample,
)
enable_x64()

channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
fit2_polar = {
    "sigma": (1.17,205.7), "rho770": (1.0,0.0), "NR": (0.48,57.3),
    "f0_980": (0.43,165.0), "f2_1270": (0.76,57.3),
    "f0_1370": (0.26,105.4), "rho1450": (0.14,319.1),
}
def polar_to_xy(r, phi):
    phi = np.deg2rad(phi)
    return r*np.cos(phi), r*np.sin(phi)
fit2_xy = {k: polar_to_xy(*v) for k,v in fit2_polar.items()}

truth = {}
def free_c(name, scale):
    xt, yt = fit2_xy[name]
    truth[f"{name}.x"], truth[f"{name}.y"] = xt, yt
    x = Parameter.coefficient(f"{name}.x", scale*xt, owner=name,
                              bounds=(-2,2), step=0.01)
    y = Parameter.coefficient(f"{name}.y", scale*yt, owner=name,
                              bounds=(-2,2), step=0.01)
    return RealImag(x,y)

rho1450_mass = Parameter.dynamics(
    "rho1450.mass", 1.40, owner="rho1450",
    bounds=(1.30,1.60), step=0.002,
)
rho1450_width = Parameter.dynamics(
    "rho1450.width", 0.36, owner="rho1450",
    bounds=(0.15,0.50), step=0.003,
)
truth["rho1450.mass"], truth["rho1450.width"] = 1.465, 0.310

c = {
    "sigma": free_c("sigma",0.80), "rho770": RealImag(1,0),
    "NR": free_c("NR",0.70), "f0_980": free_c("f0_980",0.80),
    "f2_1270": free_c("f2_1270",0.75),
    "f0_1370": free_c("f0_1370",0.70),
    "rho1450": free_c("rho1450",0.65),
}
components = [
    Resonance("sigma",(0,1),c["sigma"],mass=0.478,width=0.324,spin=0,
              resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho770",(0,1),c["rho770"],mass=0.7693,width=0.1502,spin=1,
              resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_980",(0,1),c["f0_980"],mass=0.975,width=0.044,spin=0,
              resonance_radius=3.0,parent_radius=3.0),
    Resonance("f2_1270",(0,1),c["f2_1270"],mass=1.275,width=0.185,spin=2,
              resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_1370",(0,1),c["f0_1370"],mass=1.434,width=0.173,spin=0,
              resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho1450",(0,1),c["rho1450"],mass=rho1450_mass,
              width=rho1450_width,spin=1,resonance_radius=3.0,parent_radius=3.0),
    NonResonant(c["NR"]),
]
model = DecayModel(channel, components)

# Randomize every free fit parameter inside its allowed interval.
# The fixed seed makes the notebook reproducible while keeping the start
# independent of the injected truth values.
rng = np.random.default_rng(314159)
start = {}
for parameter in model.parameters:
    if parameter.fixed:
        continue
    low, high = parameter.bounds
    start[parameter.name] = float(rng.uniform(low, high))

print("free parameters:", [p.name for p in model.parameters if not p.fixed])
print("randomized start:")
for name, value in start.items():
    print(f"  {name:14s} = {value:+.6f}")


## Generate pseudo-data and normalization MC

The target candidate weight is $w_{PS}|A(\theta_{gen})|^2$. We resample 100k unweighted pseudo-events and use an independent 1M-event normalization sample.

In [ ]:
N_POOL, N_DATA, N_NORM = 1_000_000, 100_000, 1_000_000
pool = model.generate_phase_space(N_POOL, seed=2000)
target_w = pool.weights * model.intensity(pool.as_dict(), truth)
data = weighted_resample(jax.random.key(791), pool, target_w, N_DATA, replace=True)
norm = model.generate_phase_space(N_NORM, seed=2027)
cache = model.prepare_cache(data, norm)
print("data:", data.size, " normalization:", norm.size)


In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]",
       title="E791 Fit 2 pseudo-data")
plt.show()


## Unbinned fit

$-\log\mathcal L=-\sum_n\log |A(x_n)|^2+N_{data}\log\mathcal N(\theta)$. The minimizer starts from the randomized parameter point printed above.

In [ ]:
def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity,min=1e-300))) + data.size*jnp.log(normalization)

print("NLL(randomized start):", float(nll(start)))
print("NLL(truth):", float(nll(truth)))
result = Minimizer(nll, model.parameters).fit(start_values=start)
print(result)
print("valid:", result.valid)
fit_values = {name:float(result.values[name]) for name in result.parameters}


In [ ]:
print(f"{'parameter':14s} {'gen':>10s} {'fit':>10s} {'err':>10s} {'pull':>9s} {'<1sigma':>9s}")
for p in model.parameters:
    if p.fixed: continue
    gen, fit, err = truth[p.name], float(result.values[p.name]), float(result.errors[p.name])
    pull = (gen-fit)/err
    print(f"{p.name:14s} {gen:10.5f} {fit:10.5f} {err:10.5f} {pull:9.3f} {str(abs(pull)<1):>9s}")


## Projection before and after the fit, including a zoom around the $\rho(1450)$ region.

In [ ]:
def proj(values,bins):
    w = np.asarray(norm.weights * model.intensity(norm.as_dict(), values))
    h12,_ = np.histogram(np.asarray(norm.s12),bins=bins,weights=w)
    h13,_ = np.histogram(np.asarray(norm.s13),bins=bins,weights=w)
    return h12+h13

s = np.concatenate([np.asarray(data.s12),np.asarray(data.s13)])
bins = np.linspace(s.min(),s.max(),110)
centers = 0.5*(bins[:-1]+bins[1:])
hd,_ = np.histogram(s,bins=bins)
hs,hf,ht = proj(start,bins),proj(fit_values,bins),proj(truth,bins)
for h in (hs,hf,ht): h *= hd.sum()/h.sum()

fig,ax=plt.subplots(figsize=(10,5.5))
ax.errorbar(centers,hd,yerr=np.sqrt(np.maximum(hd,1)),fmt=".",label="pseudo-data")
ax.step(centers,hs,where="mid",label="randomized start")
ax.step(centers,hf,where="mid",label="after fit")
ax.step(centers,ht,where="mid",linestyle="--",label="generated model")
ax.set(xlabel=r"$m^2(\pi^-\pi^+)$ [GeV$^2$]",ylabel="entries / bin",
       title="E791 Fit 2 projection")
ax.legend(); plt.show()

mask=(centers>1.30**2)&(centers<1.60**2)
fig,ax=plt.subplots(figsize=(9,5))
ax.errorbar(centers[mask],hd[mask],yerr=np.sqrt(np.maximum(hd[mask],1)),fmt=".",label="pseudo-data")
ax.step(centers[mask],hs[mask],where="mid",label="randomized start")
ax.step(centers[mask],hf[mask],where="mid",label="after fit")
ax.step(centers[mask],ht[mask],where="mid",linestyle="--",label="generated model")
ax.set(xlabel=r"$m^2(\pi^-\pi^+)$ [GeV$^2$]",ylabel="entries / bin",
       title=r"$\rho(1450)$-sensitive region")
ax.legend(); plt.show()
